In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import date, datetime
import pprint
import os

## 1. CVM

In [0]:
bronze_path_cvm = "/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/"

# Descobrir as partições direto no storage 
particoes = dbutils.fs.ls(bronze_path_cvm)

datas = [
    int(p.name.split('=')[1].replace('/', '')) for p in particoes if "data_processamento=" in p.name
]

if not datas:   
    print("Nenhuma partição encontrada")
    df_bronze_cvm = None
else: 
    # ultimaa partição
    ultima_particao = max(datas)
    print(f"Última partição: {ultima_particao}")

    df_bronze_cvm = spark.read.format("delta").load(bronze_path_cvm).where(f"data_processamento = {ultima_particao}")


In [0]:
distinctfundo = df_bronze_cvm.groupBy(f.col('TP_FUNDO_CLASSE')).count()
distinctfundo.show()

In [0]:
df_bronze_cvm.count()

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_bronze_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_bronze_cvm.columns
])

df_contagem_nulos.show(vertical=True)

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'VL_TOTAL']

# Aplicando a filtro para dropar as colunas
df_bronze_cvm = df_bronze_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Com a mudança de rosolução da CVM (***Resolução CVM 175***), Com a nova regra, os fundos passaram a ser estruturados em classes e subclasses, adotando o tipo "CLASSES - FIF" (Fundo de Investimento Financeiro).
Caso acha dados do mesmo ***CNPJ_FUNDO_CLASSE***, os dados de "CLASSES - FIF" terão prioridade e o evento com nomecclatura antiga será excluido.

```
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|TP_FUNDO_CLASSE| CNPJ_FUNDO_CLASSE|ID_SUBCLASSE| DT_COMPTC|   VL_TOTAL|      VL_QUOTA|VL_PATRIM_LIQ|CAPTC_DIA|RESG_DIA|NR_COTST|data_processamento|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|  CLASSES - FIF|12.586.174/0001-67|        NULL|2026-01-07|47445220.02|1.406455790000|  47448983.02|     0.00|    0.00|       1|          20260221|
|             FI|12.586.174/0001-67|        NULL|2026-01-07|47445414.65|1.406474270000|  47449606.58|     0.00|    0.00|       1|          20260221|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
```

In [0]:
# Coluna temporaria para definir prioridade em CLASSES - FIF
df_bronze_cvm = df_bronze_cvm.withColumn(
    "prioridade_tipo",
    f.when(f.col("TP_FUNDO_CLASSE") ==  "CLASSES - FIF", 1).otherwise(2)
)

# Definindo a janela  particionando pelas colunas CORE
window_spec = Window.partitionBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").orderBy("prioridade_tipo")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_bronze_cvm = df_bronze_cvm.withColumn("row_num", f.row_number().over(window_spec))

# filtrando prioridade_tipo = 1 de cada grupo e removendo as colunas auxiliares 
df_bronze_cvm = df_bronze_cvm.filter(f.col("row_num") == 1).drop("prioridade_tipo", "row_num")


#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
df_bronze_cvm = df_bronze_cvm\
    .withColumn('tp_fundo_classe', f.col('TP_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('id_subclasse', f.col('ID_SUBCLASSE').cast(t.StringType()))\
    .withColumn('dt_comptc', f.col('DT_COMPTC').cast(t.DateType()))\
    .withColumn('vl_total', f.col('VL_TOTAL').cast(t.DecimalType(38,2)))\
    .withColumn('vl_quota', f.col('VL_QUOTA').cast(t.DecimalType(38,11)))\
    .withColumn('vl_patrim_liq', f.col('VL_PATRIM_LIQ').cast(t.DecimalType(38,2)))\
    .withColumn('captc_dia', f.col('CAPTC_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('resg_dia', f.col('RESG_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('nr_cotst', f.col('NR_COTST').cast(t.LongType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

In [0]:
df_bronze_cvm.show()

### 1.2 Salvar na camada Silver

In [0]:
df_bronze_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fundos_diario")

## Bank

In [0]:
bronze_path_bank = "/Volumes/workspace/case_spark_cvm/bronze/data_bank/"

# Descobrir as partições direto no storage 
particoes = dbutils.fs.ls(bronze_path_bank)

datas = [
    int(p.name.split('=')[1].replace('/', '')) for p in particoes if "data_processamento=" in p.name
]

if not datas:   
    print("Nenhuma partição encontrada")
    df_bronze_cvm = None
else: 
    # ultimaa partição
    ultima_particao = max(datas)
    print(f"Última partição: {ultima_particao}")

    df_bronze_bank = spark.read.format("delta").load(bronze_path_bank).where(f"data_processamento = {ultima_particao}")

In [0]:
df_bronze_bank.show()

In [0]:
# Aplicando a filtro para dropar as colunaas com dados nulos 
df_bronze_bank = df_bronze_bank.dropna()

### 1.1 tratemento silver

In [0]:
df_bronze_bank = df_bronze_bank\
    .withColumn('code', f.col('code').cast(t.IntegerType()))\
    .withColumn('fullName', f.col('fullName').cast(t.StringType()))\
    .withColumn('ispb', f.col('ispb').cast(t.StringType()))\
    .withColumn('name', f.col('name').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

In [0]:
df_bronze_bank.show()

In [0]:
df_bronze_bank.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_data_bank")

## registro_classe_cvm


In [0]:
bronze_path_registro_classe_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_classe_cvm/"

# Descobrir as partições direto no storage 
particoes = dbutils.fs.ls(bronze_path_registro_classe_cvm)

datas = [
    int(p.name.split('=')[1].replace('/', '')) for p in particoes if "data_processamento=" in p.name
]

if not datas:   
    print("Nenhuma partição encontrada")
    df_bronze_cvm = None
else: 
    # ultimaa partição
    ultima_particao = max(datas)
    print(f"Última partição: {ultima_particao}")

    df_registro_classe_cvm=  spark.read.format("delta").load(bronze_path_registro_classe_cvm).where(f"data_processamento = {ultima_particao}")


In [0]:
df_registro_classe_cvm.toPandas()

### 1.1 tratemento silver

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_registro_classe_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_registro_classe_cvm.columns
])

df_contagem_nulos.show(vertical=True)

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Tipo_Classe', 'Data_Inicio','Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_classe_cvm = df_registro_classe_cvm.dropna(subset=colunas_obrigatorias)

In [0]:

df_registro_classe_cvm = df_registro_classe_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('cnpj_classe', f.col('CNPJ_Classe').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('tipo_classe', f.col('Tipo_Classe').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('classificacao', f.col('Classificacao').cast(t.StringType())) \
    .withColumn('indicador_desempenho', f.col('Indicador_Desempenho').cast(t.StringType())) \
    .withColumn('classe_cotas', f.col('Classe_Cotas').cast(t.StringType())) \
    .withColumn('classificacao_anbima', f.col('Classificacao_Anbima').cast(t.StringType())) \
    .withColumn('tributacao_longo_prazo', f.col('Tributacao_Longo_Prazo').cast(t.StringType())) \
    .withColumn('entidade_investimento', f.col('Entidade_Investimento').cast(t.StringType())) \
    .withColumn('permitido_aplicacao_cemporcento_exterior', f.col('Permitido_Aplicacao_CemPorCento_Exterior').cast(t.StringType())) \
    .withColumn('classe_esg', f.col('Classe_ESG').cast(t.StringType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('cnpj_auditor', f.col('CNPJ_Auditor').cast(t.StringType())) \
    .withColumn('auditor', f.col('Auditor').cast(t.StringType())) \
    .withColumn('cnpj_custodiante', f.col('CNPJ_Custodiante').cast(t.StringType())) \
    .withColumn('custodiante', f.col('Custodiante').cast(t.StringType())) \
    .withColumn('cnpj_controlador', f.col('CNPJ_Controlador').cast(t.StringType())) \
    .withColumn('controlador', f.col('Controlador').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


In [0]:
df_registro_classe_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_classe_cvm")

## registro_fundo_cvm

In [0]:
bronze_path_registro_fundo_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_fundo_cvm/"

# Descobrir as partições direto no storage 
particoes = dbutils.fs.ls(bronze_path_registro_fundo_cvm)

datas = [
    int(p.name.split('=')[1].replace('/', '')) for p in particoes if "data_processamento=" in p.name
]

if not datas:   
    print("Nenhuma partição encontrada")
    df_bronze_cvm = None
else: 
    # ultimaa partição
    ultima_particao = max(datas)
    print(f"Última partição: {ultima_particao}")

    df_registro_fundo_cvm = spark.read.format("delta").load(bronze_path_registro_fundo_cvm).where(f"data_processamento = {ultima_particao}")


In [0]:
df_registro_fundo_cvm.toPandas()

In [0]:
fundosditistos = df_registro_fundo_cvm.groupBy(f.col("Tipo_Fundo")).count()
fundosditistos.show()

### 1.1 tratemento silver

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_registro_fundo_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_registro_fundo_cvm.columns
])

df_contagem_nulos.show(vertical=True)

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'CNPJ_Fundo', 'Codigo_CVM', 'Tipo_Fundo', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_fundo_cvm = df_registro_fundo_cvm.dropna(subset=colunas_obrigatorias)

In [0]:
df_registro_fundo_cvm = df_registro_fundo_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('cnpj_fundo', f.col('CNPJ_Fundo').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('tipo_fundo', f.col('Tipo_Fundo').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('data_cancelamento', f.col('Data_Cancelamento').cast(t.DateType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('data_adaptacao_rcvm175', f.col('Data_Adaptacao_RCVM175').cast(t.DateType())) \
    .withColumn('data_inicio_exercicio_social', f.col('Data_Inicio_Exercicio_Social').cast(t.DateType())) \
    .withColumn('data_fim_exercicio_social', f.col('Data_Fim_Exercicio_Social').cast(t.DateType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('diretor', f.col('Diretor').cast(t.StringType())) \
    .withColumn('cnpj_administrador', f.col('CNPJ_Administrador').cast(t.StringType())) \
    .withColumn('administrador', f.col('Administrador').cast(t.StringType())) \
    .withColumn('tipo_pessoa_gestor', f.col('Tipo_Pessoa_Gestor').cast(t.StringType())) \
    .withColumn('cpf_cnpj_gestor', f.col('CPF_CNPJ_Gestor').cast(t.StringType())) \
    .withColumn('gestor', f.col('Gestor').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


In [0]:
df_registro_fundo_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_fundo_cvm")

## registro_subclasse_cvm

In [0]:
bronze_path_registro_subclasse_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_subclasse_cvm/"

# Descobrir as partições direto no storage 
particoes = dbutils.fs.ls(bronze_path_registro_subclasse_cvm)

datas = [
    int(p.name.split('=')[1].replace('/', '')) for p in particoes if "data_processamento=" in p.name
]

if not datas:   
    print("Nenhuma partição encontrada")
    df_bronze_cvm = None
else: 
    # ultimaa partição
    ultima_particao = max(datas)
    print(f"Última partição: {ultima_particao}")

    df_registro_subclasse_cvm = spark.read.format("delta").load(bronze_path_registro_subclasse_cvm).where(f"data_processamento = {ultima_particao}")



In [0]:
df_registro_subclasse_cvm.limit(2).toPandas()

### 1.1 tratemento silver

In [0]:
# Conta a quantidade de valores nulos para cada coluna do dataframe
df_contagem_nulos = df_registro_subclasse_cvm.select([
    f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_registro_subclasse_cvm.columns
])

df_contagem_nulos.show(vertical=True)

In [0]:
# Lista de colunas de caso falte dados precisamos dropala 
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Subclasse', 'Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_subclasse_cvm = df_registro_subclasse_cvm.dropna(subset=colunas_obrigatorias)

In [0]:
df_registro_subclasse_cvm = df_registro_subclasse_cvm \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('id_subclasse', f.col('ID_Subclasse').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('previdenciario', f.col('Previdenciario').cast(t.StringType())) \
    .withColumn('exclusivo_inr', f.col('Exclusivo_INR').cast(t.StringType())) \
    .withColumn('exclusivo_previdencia_complementar', f.col('Exclusivo_Previdencia_Complementar').cast(t.StringType())) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


In [0]:
df_registro_subclasse_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_subclasse_cvm")